# Case study: magical terminology in the Septuagint

This notebook uses the pinned **CenterBLC/LXX 1935 + `catss-lxx`** and **BHSA 2021 + `catss-bhsa`** datasets to reproduce the corpus-checkable layer of Maria Yurovitskaya's observations on Greek magical terminology.

It also serves as an ergonomics test for CATSS-TF. The workflow deliberately starts from ordinary Text-Fabric APIs and the current CATSS-TF feature schema rather than a case-study-specific convenience layer.

**Primary source:** Maria Yurovitskaya, “Magic in Hebrew and Greek,” Oxford Centre for Hebrew and Jewish Studies annual report 2017–2018, printed pp. 41–42: https://www.ochjs.ac.uk/wp-content/uploads/2019/02/ochjs-2018-pdf.pdf

The notebook separates direct corpus observations from semantic/historical interpretation. Claims about Classical/Hellenistic/Roman Greek outside the Septuagint require external corpora and are not tested here.

## Claims used as test cases

| Claim | What this notebook can test |
|---|---|
| `μάγος` occurs only twice, both in Daniel | occurrence count, references, CATSS Daniel source |
| `γόης` is absent | full-parent lexical lookup |
| the `φάρμακον` group behaves unusually; `φάρμακον` is plural in the LXX examples | lexical distribution, surface forms, grammatical number |
| `ἐπαοιδός` becomes a major LXX term and appears especially in Exodus / later Daniel | frequencies and book/source distribution; borrowing is interpretive |
| “passing through fire” is rendered differently in Deut 18:11; Ezek 16:21; Ezek 23:37; Jer 32:35 [LXX 39:35] | inspect Greek lexical choices and their aligned Hebrew material |

The source's explanations involving Persian topoi, snake charming, diachrony outside the LXX, or translator motivation are explicitly outside the evidence supplied by CATSS-TF.

## 1. Configuration

Materialize both CATSS-TF modules from the **same CATSS source snapshot** before running this notebook. Only the two module paths should normally need editing. No corpus data are stored in this notebook.

In [ ]:
from __future__ import annotations

import os
import unicodedata
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import display
from tf.app import use

from catss_tf.bhsa_schema import BhsaSourceStatus, classify_catss_source

LXX_MODULE = Path(os.environ.get("CATSS_LXX_MODULE", "../../generated/catss-lxx")).resolve()
BHSA_MODULE = Path(os.environ.get("CATSS_BHSA_MODULE", "../../generated/catss-bhsa")).resolve()

print("LXX module :", LXX_MODULE)
print("BHSA module:", BHSA_MODULE)


## 2. Load the pinned parent corpora plus local CATSS modules

The parent node spaces stay independent. A BHSA node number and an LXX node number must never be compared directly; cross-corpus traversal below uses `catss_alignment_id`.

In [ ]:
def load_with_module(app: str, *, checkout: str, version: str, module_dir: Path):
    if not module_dir.is_dir():
        raise FileNotFoundError(module_dir)
    return use(
        app,
        checkout=checkout,
        version=version,
        locations=str(module_dir.parent),
        modules=module_dir.name,
    )

LXX = load_with_module(
    "CenterBLC/LXX:v1.0.1",
    checkout="v1.0.1",
    version="1935",
    module_dir=LXX_MODULE,
)
BHSA = load_with_module(
    "ETCBC/bhsa:v1.8.1",
    checkout="v1.8.1",
    version="2021",
    module_dir=BHSA_MODULE,
)

for name, app in (("LXX", LXX), ("BHSA", BHSA)):
    missing = {"catss_alignment_id", "catss_source"} - set(app.api.Fall())
    if missing:
        raise RuntimeError(f"{name} is missing CATSS module features: {sorted(missing)}")


## 3. Discover lexical values rather than assuming an accent convention

CenterBLC describes `lex_utf8` as the normalized-word feature. We use its actual values and normalize only for **discovery**; all reported values remain the parent's original feature values.

In [ ]:
def plain_greek(value: str) -> str:
    decomposed = unicodedata.normalize("NFD", value)
    return "".join(ch for ch in decomposed if unicodedata.category(ch) != "Mn").casefold()

LXX_WORDS = tuple(LXX.api.F.otype.s("word"))
LXX_LEXEMES = tuple(sorted({LXX.api.F.lex_utf8.v(w) for w in LXX_WORDS if LXX.api.F.lex_utf8.v(w)}))

def find_lexemes(*normalized_stems: str) -> list[str]:
    stems = tuple(plain_greek(stem) for stem in normalized_stems)
    return [lex for lex in LXX_LEXEMES if any(plain_greek(lex).startswith(stem) for stem in stems)]

families = {
    "mag-": find_lexemes("μαγ"),
    "goet-": find_lexemes("γοη"),
    "pharmak-": find_lexemes("φαρμακ"),
    "epaoid-/epoid-": find_lexemes("επαοιδ", "επωδ"),
}
families


The inventory cell is intentionally visible. It catches spelling/normalization assumptions before they can silently turn into a zero-occurrence result.

## 4. Occurrences, morphology, and CATSS membership


In [ ]:
def feature_value(app, feature: str, node: int):
    if feature not in app.api.Fall():
        return None
    return getattr(app.api.F, feature).v(node)

def membership_rows(app, node: int) -> list[dict[str, object]]:
    rows = []
    for lane, suffix in ((1, ""), (2, "_2")):
        aid_name = f"catss_alignment_id{suffix}"
        if aid_name not in app.api.Fall():
            continue
        alignment_id = getattr(app.api.F, aid_name).v(node)
        if alignment_id is None:
            continue
        rows.append({
            "lane": lane,
            "alignment_id": alignment_id,
            "source": feature_value(app, f"catss_source{suffix}", node),
            "mapping": feature_value(app, f"catss_mapping{suffix}", node),
            "lxx_plus": feature_value(app, f"catss_lxx_plus{suffix}", node),
            "lxx_minus": feature_value(app, f"catss_lxx_minus{suffix}", node),
        })
    return rows

def lxx_occurrences(lexemes: list[str] | tuple[str, ...]) -> list[dict[str, object]]:
    rows = []
    for lexeme in lexemes:
        for node in LXX.api.F.lex_utf8.s(lexeme):
            book, chapter, verse = LXX.api.T.sectionFromNode(node)
            memberships = membership_rows(LXX, node) or [{}]
            for membership in memberships:
                rows.append({
                    "node": node,
                    "book": book,
                    "chapter": chapter,
                    "verse": verse,
                    "word": feature_value(LXX, "word", node),
                    "lex_utf8": lexeme,
                    "sp": feature_value(LXX, "sp", node),
                    "case": feature_value(LXX, "case", node),
                    "nu": feature_value(LXX, "nu", node),
                    "morphology": feature_value(LXX, "morphology", node),
                    **membership,
                })
    return rows

family_rows = {
    family: lxx_occurrences(lexemes)
    for family, lexemes in families.items()
}

summary = []
for family, rows in family_rows.items():
    summary.append({
        "family": family,
        "lexical_values": len(families[family]),
        "occurrences": len({row["node"] for row in rows}),
        "books": dict(Counter(row["book"] for row in rows)),
        "catss_sources": dict(Counter(row.get("source") for row in rows if row.get("source"))),
    })
display(pd.DataFrame(summary))


## 5. Check the narrow lexical claims

Use normalized equality only to select the intended parent lexical value. If a lookup is ambiguous, the assertion fails and the inventory above must be inspected instead of silently combining distinct lexemes.

In [ ]:
def exact_lexeme(unaccented: str) -> str | None:
    target = plain_greek(unaccented)
    matches = [lex for lex in LXX_LEXEMES if plain_greek(lex) == target]
    if not matches:
        return None
    if len(matches) != 1:
        raise ValueError(f"ambiguous normalized lexeme {unaccented!r}: {matches}")
    return matches[0]

targets = {
    "magos": exact_lexeme("μαγος"),
    "goes": exact_lexeme("γοης"),
    "pharmakon": exact_lexeme("φαρμακον"),
    "pharmakos": exact_lexeme("φαρμακος"),
    "epaoidos": exact_lexeme("επαοιδος"),
    "epoidos": exact_lexeme("επωδος"),
}
targets


In [ ]:
def rows_for_target(name: str) -> list[dict[str, object]]:
    lexeme = targets[name]
    return [] if lexeme is None else lxx_occurrences([lexeme])

magos = rows_for_target("magos")
goes = rows_for_target("goes")
pharmakon = rows_for_target("pharmakon")

checks = pd.DataFrame([
    {
        "claim": "μάγος occurs twice, both in Daniel",
        "count": len({row["node"] for row in magos}),
        "books": dict(Counter(row["book"] for row in magos)),
        "catss_sources": dict(Counter(row.get("source") for row in magos if row.get("source"))),
    },
    {
        "claim": "γόης is absent",
        "count": len({row["node"] for row in goes}),
        "books": dict(Counter(row["book"] for row in goes)),
        "catss_sources": dict(Counter(row.get("source") for row in goes if row.get("source"))),
    },
    {
        "claim": "φάρμακον number distribution",
        "count": len({row["node"] for row in pharmakon}),
        "books": dict(Counter(row["book"] for row in pharmakon)),
        "number_values": dict(Counter(row.get("nu") for row in pharmakon)),
    },
])
display(checks)
display(pd.DataFrame(magos))
display(pd.DataFrame(pharmakon))


For Daniel, do not collapse `45.DanielOG.par` and `46.DanielTh.par`. The `catss_source` column is part of the result, not incidental provenance.

## 6. Traverse Greek → CATSS alignment → Hebrew

This is the main cross-corpus operation. The only join key is the shared CATSS alignment identity.

In [ ]:
def nodes_for_alignment(app, alignment_id: str) -> tuple[int, ...]:
    nodes: set[int] = set()
    for suffix in ("", "_2"):
        name = f"catss_alignment_id{suffix}"
        if name in app.api.Fall():
            nodes.update(getattr(app.api.F, name).s(alignment_id))
    return tuple(sorted(nodes))

def first_present(app, node: int, *features: str):
    for feature in features:
        value = feature_value(app, feature, node)
        if value is not None:
            return value
    return None

def bhsa_join_status(source: str | None, lxx_plus: object, nodes: tuple[int, ...]) -> str:
    if nodes:
        return "mapped"
    if lxx_plus:
        return "lxx_plus_no_hebrew_word"
    if source:
        profile = classify_catss_source(source)
        if profile.status is not BhsaSourceStatus.SUPPORTED:
            return "no_bhsa_projection"
    return "no_bhsa_word_mapping"

def greek_to_hebrew(rows: list[dict[str, object]]) -> list[dict[str, object]]:
    joined = []
    for row in rows:
        alignment_id = row.get("alignment_id")
        if not alignment_id:
            joined.append({**row, "bhsa_status": "no_catss_membership"})
            continue
        hebrew_nodes = nodes_for_alignment(BHSA, str(alignment_id))
        status = bhsa_join_status(row.get("source"), row.get("lxx_plus"), hebrew_nodes)
        if not hebrew_nodes:
            joined.append({**row, "bhsa_status": status})
            continue
        for hnode in hebrew_nodes:
            hbook, hchapter, hverse = BHSA.api.T.sectionFromNode(hnode)
            joined.append({
                **row,
                "bhsa_status": status,
                "bhsa_node": hnode,
                "bhsa_book": hbook,
                "bhsa_chapter": hchapter,
                "bhsa_verse": hverse,
                "hebrew_word": first_present(BHSA, hnode, "g_word_utf8", "qere_utf8", "g_cons_utf8"),
                "hebrew_lex": first_present(BHSA, hnode, "lex_utf8", "lex"),
                "hebrew_sp": feature_value(BHSA, "sp", hnode),
            })
    return joined


In [ ]:
epaoid_rows = family_rows["epaoid-/epoid-"]
epaoid_joined = greek_to_hebrew(epaoid_rows)

cols = [
    "book", "chapter", "verse", "word", "lex_utf8",
    "source", "alignment_id", "bhsa_status",
    "hebrew_word", "hebrew_lex", "bhsa_book", "bhsa_chapter", "bhsa_verse",
]
display(pd.DataFrame(epaoid_joined).reindex(columns=cols))


This table is the empirical basis for asking whether the same Greek term is repeatedly selected for the same Hebrew vocabulary across books. A historical claim that a later translator *borrowed* Pentateuchal usage needs argument beyond the join itself.

## 7. Four “passing through fire” passages

Jeremiah is deliberately represented with different MT/BHSA and LXX references. This is a useful test of whether a workflow accidentally assumes identical versification.

In [ ]:
PASSAGES = [
    {"label": "Deut 18:11", "lxx": ("Deut", 18, 11), "bhsa": ("Deuteronomium", 18, 11)},
    {"label": "Ezek 16:21", "lxx": ("Ezek", 16, 21), "bhsa": ("Ezechiel", 16, 21)},
    {"label": "Ezek 23:37", "lxx": ("Ezek", 23, 37), "bhsa": ("Ezechiel", 23, 37)},
    {"label": "Jer 32:35 [LXX 39:35]", "lxx": ("Jer", 39, 35), "bhsa": ("Jeremia", 32, 35)},
]

def verse_words(app, section: tuple[object, ...]) -> tuple[int, ...]:
    verse_node = app.api.T.nodeFromSection(section)
    if verse_node is None:
        raise LookupError(section)
    return tuple(app.api.L.d(verse_node, otype="word"))

def lxx_verse_rows(section: tuple[object, ...]) -> list[dict[str, object]]:
    rows = []
    for node in verse_words(LXX, section):
        memberships = membership_rows(LXX, node) or [{}]
        for membership in memberships:
            rows.append({
                "node": node,
                "book": section[0],
                "chapter": section[1],
                "verse": section[2],
                "word": feature_value(LXX, "word", node),
                "lex_utf8": feature_value(LXX, "lex_utf8", node),
                **membership,
            })
    return rows

for passage in PASSAGES:
    print("\n", passage["label"])
    paired = greek_to_hebrew(lxx_verse_rows(passage["lxx"]))
    display(pd.DataFrame(paired).reindex(columns=cols))


Read these verse tables philologically. CATSS-TF exposes the lexical/alignment evidence and provenance; the classification of a rendering as “magical” or “alien religious practice” remains an interpretation.

## 8. What the corpus cannot decide

- Whether `ἐπαοιδός` was selected because of snake-charming associations.
- Whether later translators consciously borrowed the word from the Pentateuch.
- Whether `φάρμακος` is a Septuagintal neologism or how early it occurs outside Jewish-Hellenistic literature.
- How frequent these roots are in Classical and Hellenistic-Roman Greek generally.

Those are natural next steps using external Greek corpora, papyri/inscriptions, lexica, and dated literary evidence.

## 9. Ergonomics audit

After running the notebook on real data, record concrete friction here and convert generally useful gaps into focused issues.

- **Setup:** are two parent apps + two local modules easy to configure reproducibly?
- **Lexical discovery:** is the meaning/naming of `lex_utf8` obvious enough for a new user?
- **Membership lanes:** does research code have to know about `_2`, or should CATSS-TF expose a helper iterator?
- **Cross-corpus join:** is repeatedly implementing `alignment_id -> nodes` excessive boilerplate?
- **Coverage state:** can a user readily distinguish LXX-plus, unsupported BHSA projection, and a suspicious missing mapping?
- **Versification:** can divergent MT/LXX references be inspected without manual special cases?
- **Presentation:** how much code is required to obtain a scholar-readable Greek ↔ Hebrew table?

The goal is to improve reusable corpus ergonomics where the notebook demonstrates a real recurring need, not to build a bespoke API around this one paper.

## 10. Further directions

1. Expand from the four word families to a researcher-defined semantic field and compute Hebrew→Greek and Greek→Hebrew correspondence distributions.
2. Compare Pentateuch, Prophets, Writings, Daniel OG, and Theodotion separately rather than treating “the Septuagint” as one translator.
3. Combine the lexical correspondences with CATSS translation-technique features (plus/minus, cardinality, transposition) to ask whether these terms occur in structurally marked renderings.
4. Add an external Greek corpus to test the comparative/diachronic claims that CATSS-TF cannot address.
5. If the same workflow is useful across case studies, factor only the demonstrated general operations into a small query helper API with tests.